# Model Inference

## Objective

This notebook demonstrates how to use the trained machine learning pipeline for predicting whether a new patient has a disease.

The workflow includes:

- Loading the trained pipeline
- Creating new patient records
- Making predictions
- Predicting probabilities
- Displaying human-readable results

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

In [2]:
pipeline = joblib.load("../artifacts/best_pipeline.joblib")

print("Pipeline Loaded Successfully")

Pipeline Loaded Successfully


In [3]:
df = pd.read_csv("../data/Patient_Health_Records_Raw.csv")

df.head()

,Patient_ID,Name,Age,Gender,City,BMI,Blood_Pressure,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Last_Visit_Date,Follow_Up,Diagnosis_Code,Notes,Has_Disease
0,ID-00001,"SMITH, ALICE",250,Unknown,Boston,22kg/m2,120 over 80,83,190,Unknown,yes,NaN,Apr 15 2021,30,A00,Patient feeling fine 🙂,0
1,ID-00002,"SMITH, ALICE",-5,FEMALE,newyork,25.8,NaN,500,normal,N,No,NaN,2021-07-10,NaN,unknown,"BP high, needs recheck",unknown
2,ID-00003,John Doe,-5,NaN,Nwe Yrok,22kg/m2,120 - 80,500,190,no,Former,"aspirin,metformin",Apr 06 2021,30,B20,NaN,unknown
3,ID-00004,michael brown,80,F,New York,33.4,NaN,eighty,250,y,EX-smoker,NaN,20241113,30,NaN,Follow up soon,unknown
4,ID-00005,Jane Doe,250,male,la,30.1,120 over 80,0,normal,no,No,NaN,2025-09-27,NaN,I10,Patient feeling fine 🙂,1


In [4]:
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
)

In [5]:
df = df.drop_duplicates()

In [6]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [7]:
numeric_cols = [
    "Age",
    "BMI",
    "Blood_Pressure",
    "Heart_Rate",
    "Cholesterol_Level"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [8]:
df["Gender"] = (
    df["Gender"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "m":"male",
        "f":"female",
        "unknown":np.nan,
        "nan":np.nan
    })
)

In [9]:
df["City"] = (
    df["City"]
    .astype(str)
    .str.strip()
    .str.lower()
)

city_map = {
    "ny":"new york",
    "newyork":"new york",
    "nwe yrok":"new york",
    "la":"los angeles",
    "chicago":"chicago",
    "nan":np.nan
}

df["City"] = df["City"].replace(city_map)

In [10]:
df["Diagnosis_Code"] = df["Diagnosis_Code"].replace(
    "unknown",
    np.nan
)

In [11]:
df["Follow_Up"] = (
    df["Follow_Up"]
    .astype(str)
    .str.lower()
    .replace({
        "two weeks":"14",
        "nan":np.nan
    })
)

In [12]:
df["Medications"] = (
    df["Medications"]
    .astype(str)
    .str.lower()
    .replace({
        "aspirin,metformin":"aspirin + metformin",
        "aspirin; metformin":"aspirin + metformin",
        "lisinopril / metformin":"lisinopril + metformin",
        "nan":np.nan
    })
)

In [13]:
binary_map = {
    "yes":1,
    "no":0,
    "true":1,
    "false":0,
    "1":1,
    "0":0,
    "y":1,
    "n":0,
    "unknown":np.nan,
    "nan":np.nan
}

for col in ["Diabetic","Smoker"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .map(binary_map)
    )

In [14]:
drop_columns = [
    "Patient_ID",
    "Name",
    "Last_Visit_Date",
    "Blood_Pressure",
    "Visit_Year",
    "Visit_Month",
    "Visit_Day",
    "Notes"
]

df.drop(
    columns=drop_columns,
    inplace=True,
    errors="ignore"
)

In [15]:
if "Has_Disease" in df.columns:
    df = df.drop(columns=["Has_Disease"])

In [16]:
predictions = pipeline.predict(df)

probabilities = pipeline.predict_proba(df)[:,1]

In [17]:
results = df.copy()

results["Prediction"] = predictions

results["Probability"] = probabilities

results.head()

,Age,Gender,City,BMI,Heart_Rate,Cholesterol_Level,Diabetic,Smoker,Medications,Follow_Up,Diagnosis_Code,Prediction,Probability
0,250.0,NaN,boston,NaN,83.0,190.0,NaN,1.0,NaN,30,A00,0,0.345525
1,-5.0,female,new york,25.8,500.0,NaN,0.0,0.0,NaN,NaN,NaN,0,0.351991
2,-5.0,NaN,new york,NaN,500.0,190.0,0.0,NaN,aspirin + metformin,30,B20,0,0.425852
3,80.0,female,new york,33.4,NaN,250.0,1.0,NaN,NaN,30,nan,1,0.561224
4,250.0,male,los angeles,30.1,0.0,NaN,0.0,0.0,NaN,NaN,I10,1,0.739527


In [18]:
results.to_csv(
    "../artifacts/predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Predictions saved successfully.


In [19]:
sample = df.iloc[[0]]

prediction = pipeline.predict(sample)[0]

probability = pipeline.predict_proba(sample)[0,1]

print("Prediction:", prediction)

print("Probability:", round(probability,4))

Prediction: 0
Probability: 0.3455


In [20]:
print("="*50)
print("Inference Completed Successfully")
print("="*50)
print(results.head())

Inference Completed Successfully
     Age  Gender         City   BMI  Heart_Rate  Cholesterol_Level  Diabetic  \
0  250.0     NaN       boston   NaN        83.0              190.0       NaN   
1   -5.0  female     new york  25.8       500.0                NaN       0.0   
2   -5.0     NaN     new york   NaN       500.0              190.0       0.0   
3   80.0  female     new york  33.4         NaN              250.0       1.0   
4  250.0    male  los angeles  30.1         0.0                NaN       0.0   

   Smoker          Medications Follow_Up Diagnosis_Code  Prediction  \
0     1.0                  NaN        30            A00           0   
1     0.0                  NaN       NaN            NaN           0   
2     NaN  aspirin + metformin        30            B20           0   
3     NaN                  NaN        30            nan           1   
4     0.0                  NaN       NaN            I10           1   

   Probability  
0     0.345525  
1     0.351991  
2     0.

# Inference Summary

The trained machine learning pipeline successfully predicts disease status for new patient records.

The same preprocessing pipeline used during training is automatically applied during inference, ensuring consistency and preventing preprocessing mismatches.

Outputs include:

- Predicted Class
- Prediction Probability
- Batch Predictions
- Exported Prediction CSV

This notebook demonstrates a complete inference workflow suitable for deployment.